# Реализация ядра системы антифрода для предотвращения мошеннических операций

**Описание исследования**

Необходимо разработать алгоритм классификации банковских операций, которые по соображениям безопасности не были подтверждены клиентами — ядро системы антифрода, работающей в режиме реального времени. Для обучения предоставляется история операций 100 000 клиентов за полтора года, суммарным объёмом более 200 миллионов операций.

---

**Цель исследования**

Разработать и обучить модель бинарной классификации операций на мошеннические 🔴 и легитимные 🟢 с предсказанием вещественного скора для 633 683 операций тестового периода.

---

**Задачи исследования**

- **Анализ и подготовка данных:** изучение структуры временных срезов, обработка пропусков, кодирование категориальных признаков.
- **EDA:** исследование поведенческих паттернов клиентов по каналам, суммам, времени и устройствам.
- **Feature Engineering:** клиентские профили по Pre-train периоду, скользящие агрегаты, признаки отклонения от нормы.
- **Anomaly Detection:** обучение Isolation Forest на Pre-train для получения скоров аномальности как дополнительных признаков.
- **Обучение с учителем:** бейслайн на логистической регрессии, основная модель на LightGBM / CatBoost.
- **Валидация:** строго временной разрез, метрика — PR-AUC (`average_precision_score`).

---

**Описание данных**

| Период | Даты | Описание |
|--------|------|----------|
| Pre-train | 01.10.2023 – 30.09.2024 | История «чистого» поведения. Разметки нет. Используется для Feature Engineering и Anomaly Detection. |
| Train | 01.10.2024 – 31.05.2025 | Период с мошенническими операциями. Разметка: 🔴 не подтверждено, 🟡 подтверждено после подозрения, 🟢 остальное. |
| Pre-test | 01.06.2025 – 09.08.2025 | Контекст перед тестом. Разметки нет. Для актуализации профиля клиента. |
| Test | 01.06.2025 – 09.08.2025 | Финальный день операций каждого клиента. Требуется классифицировать. |

Каждая операция описывается следующими признаками:

| Признак | Описание |
|---------|----------|
| `customer_id` | Идентификатор клиента |
| `event_id` | Идентификатор операции |
| `event_dttm` | Дата и время операции |
| `event_type_nm` | Тип операции |
| `event_desc` | Закодированное описание операции |
| `channel_indicator_type` | Канал совершения операции |
| `channel_indicator_subtype` | Подтип канала |
| `operaton_amt` | Сумма операции в рублях |
| `currency_iso_cd` | Валюта операции |
| `mcc_code` | Группа MCC (merchant category code) |
| `pos_cd` | Закодированный point of sale condition code |
| `accept_language` | Язык заголовка HTTP-запроса |
| `browser_language` | Язык браузера |
| `timezone` | Часовой пояс устройства |
| `session_id` | Идентификатор сессии |
| `operating_system_type` | Закодированный тип операционной системы |
| `battery` | Заряд устройства в момент операции |
| `device_system_version` | Версия операционной системы |
| `screen_size` | Разрешение экрана |
| `developer_tools` | Флаг настроек разработчика на устройстве |
| `phone_voip_call_state` | Флаг VoIP-звонка во время операции |
| `web_rdp_connection` | Флаг удалённого управления устройством |
| `compromised` | Флаг наличия Root-доступа на устройстве |

## Загрузка необходимых библиотек

In [1]:
from pathlib import Path

req_path = Path("../../requirements.txt")
if req_path.exists():
    %pip install -qr {req_path}
else:
    %pip install -qU "numpy<2.2.0" "pandas==2.2.2" numba scipy scikit-learn imbalanced-learn matplotlib seaborn phik joblib category_encoders optuna plotly nbformat jinja2 catboost xgboost lightgbm shap mlxtend

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 5.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 74.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 96.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 20.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 111.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 108.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.9/85.9 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 30.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 108.7 MB/s eta 0:00:0000:0

In [2]:
import joblib
import sys

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from phik import phik_matrix

import optuna

from mlxtend.evaluate.time_series import GroupTimeSeriesSplit

import sklearn
from sklearn.model_selection import train_test_split, cross_validate, KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error, make_scorer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestRegressor, StackingRegressor

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from shap import force_plot, summary_plot, TreeExplainer, LinearExplainer, sample

from IPython.display import display, Markdown

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [3]:
RANDOM_STATE = 42

## Загрузка данных

In [4]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive монтирован")
except ImportError:
    print("Локальное использование")

Mounted at /content/drive
Google Drive монтирован


In [12]:
def get_df(file_name, read_func):
    drive_path = Path(f'/content/drive/MyDrive/guard_datasets/{file_name}')

    if drive_path.exists():
        df = read_func(drive_path)
    else:
        raise "Путь до датасета неверный"

    int_cols = df.select_dtypes(include=['integer']).columns
    for col in int_cols:
        df[col] = pd.to_numeric(df[col], downcast='integer')

    float_cols = df.select_dtypes(include=['float']).columns
    for col in float_cols:
        df[col] = pd.to_numeric(df[col], downcast='float')

    print('-' * 70)
    print(file_name)
    display(df.head())
    df.info()

    return df

sample_submit = get_df('sample_submit.csv', pd.read_csv)
train_labels = get_df('train_labels.parquet', pd.read_parquet)

pretrain_part_1 = get_df('pretrain_part_1.parquet', pd.read_parquet)
pretrain_part_2 = get_df('pretrain_part_2.parquet', pd.read_parquet)
pretrain_part_3 = get_df('pretrain_part_3.parquet', pd.read_parquet)

train_part_1 = get_df('train_part_1.parquet', pd.read_parquet)
train_part_2 = get_df('train_part_2.parquet', pd.read_parquet)
train_part_3 = get_df('train_part_3.parquet', pd.read_parquet)

pretest = get_df('pretest.parquet', pd.read_parquet)
test = get_df('test.parquet', pd.read_parquet)

----------------------------------------------------------------------
sample_submit.csv


,event_id,predict
0,125854726334416,-0.98
1,125949211749418,-1.14
2,124437385134670,-0.87
3,124394437682654,-0.87
4,123973531121838,-1.04


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 633683 entries, 0 to 633682
Data columns (total 2 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   event_id  633683 non-null  int64  
 1   predict   633683 non-null  float32
dtypes: float32(1), int64(1)
memory usage: 7.3 MB
----------------------------------------------------------------------
train_labels.parquet


,customer_id,event_id,target
0,123123123123129,124093788813382,0
1,123123123123138,126035112904381,0
2,123123123123169,124325714018852,1
3,123123123123169,124944191029616,1
4,123123123123169,126395888234936,1


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 87514 entries, 0 to 87513
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  87514 non-null  int64
 1   event_id     87514 non-null  int64
 2   target       87514 non-null  int8 
dtypes: int64(2), int8(1)
memory usage: 1.4 MB
----------------------------------------------------------------------
pretrain_part_1.parquet


,customer_id,event_id,event_dttm,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,accept_language,browser_language,timezone,session_id,operating_system_type,battery,device_system_version,screen_size,developer_tools,phone_voip_call_state,web_rdp_connection,compromised
0,123123123123129,123251972261925,2023-10-01 09:16:41,7,56,4,15,NaN,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None,None,None,NaN,NaN,None
1,123123123123129,125193298164858,2023-10-01 11:13:48,14,75,6,5,71085.00,0.00,15,8.00,None,None,NaN,NaN,NaN,None,None,None,None,NaN,NaN,None
2,123123123123129,124823932244729,2023-10-01 11:40:35,14,75,6,5,36158.00,0.00,14,8.00,None,None,NaN,NaN,NaN,None,None,None,None,NaN,NaN,None
3,123123123123129,124454563399321,2023-10-01 16:37:32,14,75,6,5,81971.00,0.00,4,8.00,None,None,NaN,NaN,NaN,None,None,None,None,NaN,NaN,None
4,123123123123129,126490377886229,2023-10-02 19:28:25,7,56,4,15,NaN,NaN,None,NaN,None,None,NaN,NaN,NaN,None,None,None,None,NaN,NaN,None


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30329960 entries, 0 to 30329959
Data columns (total 23 columns):
 #   Column                      Dtype  
---  ------                      -----  
 0   customer_id                 int64  
 1   event_id                    int64  
 2   event_dttm                  object 
 3   event_type_nm               int8   
 4   event_desc                  int16  
 5   channel_indicator_type      int8   
 6   channel_indicator_sub_type  int8   
 7   operaton_amt                float64
 8   currency_iso_cd             float32
 9   mcc_code                    object 
 10  pos_cd                      float32
 11  accept_language             object 
 12  browser_language            object 
 13  timezone                    float32
 14  session_id                  float32
 15  operating_system_type       float32
 16  battery                     object 
 17  device_system_version       object 
 18  screen_size                 object 
 19  developer_tools    

: 

: 

: 